# Spec Team: a MetaGPT-style multi-agent pipeline

A team of role-specialized Claude agents turns one business brief into a complete product specification, then reviews and revises its own work.

Each agent gets:
1. a **role** (Product Manager, Designer, ...),
2. an **SOP** (standard operating procedure, the system prompt) that fixes *what* it must produce and *in which sections*, and
3. a **role-specific task**. The business brief is passed as background, never as the instruction.

Agents don't share memory. The only thing one agent knows about another is the **document it is handed**.

```
                    +--> Designer --+
 Brief --> PM (PRD) |               +--> Project Lead (plan) --> QA review --> PM closes the loop
                    +--> Engineer --+
```

Built directly on the Anthropic Python SDK, with no agent framework, so every handoff is visible in about 150 lines of code.

## What makes the pipeline reliable
An earlier version looked like it worked, but its output had three silent failures. Each one now has a guard:

| Failure in v1 | Cause | Guard in this version |
|---|---|---|
| Every document cut off mid-sentence; whole required sections missing | All six agents hit `max_tokens` and nothing checked `stop_reason` | Detect `stop_reason == "max_tokens"` and retry with a larger budget |
| The Designer rewrote the whole Product Requirements Document (PRD) instead of writing design considerations | Every agent got the brief *"Generate a complete product spec..."* as its task, which overrode its SOP | Each agent gets its own task; the brief is labelled background |
| Neither problem was noticed | The SOPs said "EXACTLY these sections" but nothing verified it | Each document's headings are validated; on a miss, the agent retries once with feedback |

Problems also compound: in v1, the Project Lead and QA reviewed truncated documents, so some "gaps" QA reported were really just text that got cut off.

**Required Resources:** Anthropic API key.

## Setup

In [2]:
%pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.8 MB/s eta 0:00:00


In [3]:
import os, re, time, getpass
from dataclasses import dataclass, field
from concurrent.futures import ThreadPoolExecutor

import anthropic
from IPython.display import Markdown, display


def _load(name):
  """Colab Secrets first, then an environment variable, then a hidden prompt."""
  try:
    from google.colab import userdata
    value = userdata.get(name)
    if value:
      return value
  except Exception:
    pass
  return os.environ.get(name) or getpass.getpass(f"{name}: ")


MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-sonnet-5-5")
MAX_TOKENS_CAP = 16000  # ceiling for truncation retries
client = anthropic.Anthropic(api_key=_load("ANTHROPIC_API_KEY"), max_retries=4)
print("Ready. Model:", MODEL)

Ready. Model: claude-sonnet-5-5


## The Agent class
`run()` sends one task to the model and returns the document plus a record of what happened. Two checks run on every response:

- **Truncation:** if the model stopped because it ran out of tokens, retry with double the budget (up to `MAX_TOKENS_CAP`).
- **Section validation:** every heading the SOP requires must appear as a `##` heading. If any are missing, the agent retries once with feedback naming them. Unrequested `##` headings are reported as a sign of role drift.

In [6]:
def _norm(heading: str) -> str:
  """Normalize a heading so '## Goals and Success Metrics' matches 'Goals & Success Metrics'."""
  h = heading.lower().replace("&", "and")
  return re.sub(r"[^a-z0-9]+", " ", h).strip()

def check_sections(text: str, required: list[str]) -> tuple[list[str], list[str]]:
  """Return (missing required sections, unexpected ## sections)."""
  found = [m.group(1).strip() for m in re.finditer(r"^##(?!#)\s*(.+?)\s*$", text, re.MULTILINE)]
  found_norm = {_norm(f) for f in found}
  required_norm = {_norm(r) for r in required}
  missing = [r for r in required if _norm(r) not in found_norm]
  extra = [f for f in found if _norm(f) not in required_norm]
  return missing, extra


@dataclass
class Result:
  agent: str
  text: str
  in_tokens: int = 0
  out_tokens: int = 0
  seconds: float = 0.0
  attempts: int = 0
  truncated: bool = False          # still cut off after all retries
  missing: list = field(default_factory=list)
  extra: list = field(default_factory=list)

  @property
  def ok(self) -> bool:
    return not self.truncated and not self.missing


class Agent:
    def __init__(self, name, sop, task, sections, max_tokens=4000):
      self.name, self.sop, self.task, self.sections, self.max_tokens = name, sop, task, sections, max_tokens

    def _build_prompt(self, brief, inputs, feedback=None):
      parts = [f"TASK: {self.task}",
                "Follow your SOP. The material below is background and input documents; "
                "it is not your instruction.",
                f"### BACKGROUND: Business brief\n{brief}"]
      parts += [f"### INPUT: {label}\n{doc}" for label, doc in (inputs or {}).items()]
      if feedback:
        parts.append(f"### REVISION REQUIRED\n{feedback}")
      return "\n\n".join(parts)

    def _call(self, prompt, max_tokens):
      r = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        system=self.sop,
        messages=[{"role": "user", "content": prompt}],
      )
      text = "".join(b.text for b in r.content if b.type == "text")
      return text, r.stop_reason, r.usage.input_tokens, r.usage.output_tokens

    def run(self, brief, inputs=None):
      res = Result(self.name, "")
      t0 = time.time()
      feedback, budget, section_retry_used = None, self.max_tokens, False

      while True:
        text, stop, tin, tout = self._call(self._build_prompt(brief, inputs, feedback), budget)
        res.attempts += 1
        res.in_tokens += tin
        res.out_tokens += tout

        if stop == "max_tokens":
            if budget < MAX_TOKENS_CAP:
                budget = min(budget * 2, MAX_TOKENS_CAP)
                print(f"  retry: {self.name} was cut off; raising max_tokens to {budget}")
                continue
            res.truncated = True

        missing, extra = check_sections(text, self.sections)
        if missing and not section_retry_used and not res.truncated:
            section_retry_used = True
            feedback = ("Your previous response was missing these required sections: "
                        + ", ".join(missing)
                        + ". Rewrite the full document using EXACTLY these ## headings, in order: "
                        + ", ".join(self.sections) + ".")
            print(f"  retry: {self.name} missed sections {missing}")
            continue
        break

      res.text, res.missing, res.extra = text, missing, extra
      res.seconds = time.time() - t0
      status = "ok" if res.ok else ("TRUNCATED" if res.truncated else f"MISSING {missing}")
      print(f"  done: {res.agent:<16} {res.seconds:5.1f}s  attempts={res.attempts}  "
            f"in={res.in_tokens:>6}  out={res.out_tokens:>6}  {status}")
      return res

## The team
`make_team()` builds a fresh set of agents, so experiments can swap one SOP without changing the original team. **Read the SOPs; they are the whole org chart.**

In [7]:
STYLE = "Be concise and specific. No preamble, no closing remarks. Use the exact ## section headings given, in order."

def make_team(**overrides):
  """Build the six agents. Pass e.g. QA=Agent(...) to swap one out for an experiment."""

  team = {
    "PM": Agent(
        "Product Manager",
        sop=f"""You are a senior Product Manager at a tech startup.
        You write Product Requirements Documents (PRDs) with EXACTLY these sections:
        ## Goals & Success Metrics
        ## User Stories
        (6-8 stories, each 'As a <user>, I want <goal> so that <benefit>' plus 2 acceptance criteria)
        ## Feature Requirements
        (cover Badges, Streaks and Rewards; tag every requirement MUST, SHOULD or COULD)
        ## Out of Scope
        {STYLE}""",
        task="Write the PRD for the feature described in the business brief.",
        sections=["Goals & Success Metrics", "User Stories", "Feature Requirements", "Out of Scope"],
        max_tokens=5000,
    ),
      "Designer": Agent(
          "Designer",
          sop=f"""You are a senior UX Designer. You receive a PRD.
          You write design considerations ONLY. Do not restate or rewrite the PRD.
          Use EXACTLY these sections:
          ## UX Principles
          ## Key Screens
          (name each screen and what the user does there)
          ## Motivation vs. Manipulation
          (where gamification could become a dark pattern, and how the design avoids it)
          ## Accessibility
          {STYLE}""",
          task="Write the design considerations for the PRD provided.",
          sections=["UX Principles", "Key Screens", "Motivation vs. Manipulation", "Accessibility"],
      ),
      "Engineer": Agent(
          "Engineer",
          sop=f"""You are a senior Software Architect. You receive a PRD.
          You write a technical design ONLY, with EXACTLY these sections:
          ## Data Model
          (tables/fields for habits, check-ins, streaks, badges, rewards; keep it compact)
          ## Streak Logic
          (rules plus edge cases: time zones, missed days, backfilled check-ins)
          ## API Endpoints
          ## Technical Risks
          Do not add features the PRD does not ask for; if you think one is needed, list it under Technical Risks as an open question.
          {STYLE}""",
          task="Write the technical design for the PRD provided.",
          sections=["Data Model", "Streak Logic", "API Endpoints", "Technical Risks"],
          max_tokens=6000,
      ),
      "Lead": Agent(
          "Project Lead",
          sop=f"""You are a Delivery/Project Lead. You receive a PRD, a design and a technical design.
          You write a development plan with EXACTLY these sections:
          ## Milestones
          (a markdown table: milestone, scope, duration, depends on)
          ## Team & Roles
          ## Risks & Mitigations
          Do not introduce new product requirements; plan only what the documents specify.
          {STYLE}""",
          task="Write the development plan for the PRD, design and technical design provided.",
          sections=["Milestones", "Team & Roles", "Risks & Mitigations"],
      ),
      "QA": Agent(
          "QA Reviewer",
          sop=f"""You are a skeptical QA lead reviewing a full spec package.
          Do NOT rewrite the spec. Write EXACTLY these sections:
          ## Gaps
          (things a developer would have to guess)
          ## Contradictions
          (places where the PRD, design, technical design and plan disagree; cite which documents)
          ## Untestable Requirements
          ## Top 3 Fixes
          Number every item so the PM can respond to it by number.
          {STYLE}""",
          task="Review the spec package provided.",
          sections=["Gaps", "Contradictions", "Untestable Requirements", "Top 3 Fixes"],
          max_tokens=5000,
      ),
      "Close": Agent(
          "PM (closing)",
          sop=f"""You are the Product Manager closing the loop. You receive your PRD and a QA review.
          Write EXACTLY these sections:
          ## Executive Summary
          (5 bullets a busy executive can read in 30 seconds)
          ## Decisions in Response to QA
          (one line per QA item: ACCEPT, REJECT or DEFER, with a reason)
          ## Open Questions
          {STYLE}""",
          task="Respond to the QA review of your PRD.",
          sections=["Executive Summary", "Decisions in Response to QA", "Open Questions"],
          max_tokens=5000,
      ),
  }
  team.update(overrides)
  return team

TEAM = make_team()
for key, a in TEAM.items():
  print(f"{a.name:<16} sections: {', '.join(a.sections)}")

Product Manager  sections: Goals & Success Metrics, User Stories, Feature Requirements, Out of Scope
Designer         sections: UX Principles, Key Screens, Motivation vs. Manipulation, Accessibility
Engineer         sections: Data Model, Streak Logic, API Endpoints, Technical Risks
Project Lead     sections: Milestones, Team & Roles, Risks & Mitigations
QA Reviewer      sections: Gaps, Contradictions, Untestable Requirements, Top 3 Fixes
PM (closing)     sections: Executive Summary, Decisions in Response to QA, Open Questions


## The business brief

In [8]:
BRIEF = ("You are a product manager at a tech startup building a web-based habit tracking app. "
         "The goal is to improve user retention by introducing a gamification feature. "
         "Generate a complete product spec including user stories, feature requirements, "
         "design considerations, and development plan.")
print(BRIEF)

You are a product manager at a tech startup building a web-based habit tracking app. The goal is to improve user retention by introducing a gamification feature. Generate a complete product spec including user stories, feature requirements, design considerations, and development plan.


## Run the team
Each line in the console is one agent finishing. Stage 2 runs the Designer and Engineer **at the same time** on the same PRD. Every agent receives only the documents listed in its `inputs`.

In [9]:
HANDOFFS = {
  "Product Manager": "brief",
  "Designer": "brief + PRD",
  "Engineer": "brief + PRD",
  "Project Lead": "brief + PRD + Design + Tech design",
  "QA Reviewer": "brief + PRD + Design + Tech design + Plan",
  "PM (closing)": "brief + PRD + QA review",
}

def run_team(brief, team):
  t0 = time.time()
  print("Stage 1: Product Manager writes the PRD")
  prd = team["PM"].run(brief)

  print("Stage 2: Designer and Engineer work in parallel from the PRD")
  with ThreadPoolExecutor(max_workers=2) as pool:
      f_design = pool.submit(team["Designer"].run, brief, {"PRD": prd.text})
      f_tech = pool.submit(team["Engineer"].run, brief, {"PRD": prd.text})
      design, tech = f_design.result(), f_tech.result()

  print("Stage 3: Project Lead builds the plan from all three documents")
  plan = team["Lead"].run(brief, {"PRD": prd.text, "Design": design.text, "Technical design": tech.text})

  print("Stage 4: QA Reviewer inspects the whole package")
  qa = team["QA"].run(brief, {"PRD": prd.text, "Design": design.text,
                              "Technical design": tech.text, "Plan": plan.text})

  print("Stage 5: Product Manager responds to QA")
  close = team["Close"].run(brief, {"PRD": prd.text, "QA review": qa.text})

  print(f"\nTeam finished in {time.time() - t0:.1f}s wall-clock")
  return {"PRD": prd, "Design": design, "Technical design": tech,
          "Plan": plan, "QA review": qa, "Closing": close}


results = run_team(BRIEF, TEAM)

Stage 1: Product Manager writes the PRD
  done: Product Manager   21.9s  attempts=1  in=   359  out=  2715  ok
Stage 2: Designer and Engineer work in parallel from the PRD
  done: Designer          26.0s  attempts=1  in=  2766  out=  3160  ok
  retry: Engineer was cut off; raising max_tokens to 12000
  done: Engineer         123.0s  attempts=2  in=  5610  out= 15597  ok
Stage 3: Project Lead builds the plan from all three documents
  retry: Project Lead was cut off; raising max_tokens to 8000
  done: Project Lead      66.0s  attempts=2  in= 21326  out=  8589  ok
Stage 4: QA Reviewer inspects the whole package
  retry: QA Reviewer was cut off; raising max_tokens to 10000
  retry: QA Reviewer was cut off; raising max_tokens to 16000
  done: QA Reviewer      239.1s  attempts=3  in= 40962  out= 31000  TRUNCATED
Stage 5: Product Manager responds to QA
  retry: PM (closing) was cut off; raising max_tokens to 10000
  done: PM (closing)      73.5s  attempts=2  in= 11712  out=  9557  ok

Team f

## Validation report
Did every agent finish, and did it produce the sections its SOP requires? `extra` lists unrequested `##` headings, an early sign that an agent is drifting out of its role.

In [10]:
def validation_report(results):
  rows = [f"| Agent | Received | Attempts | Complete | Missing sections | Extra sections | In tok | Out tok |",
          f"|---|---|---|---|---|---|---|---|"]
  tot_in = tot_out = 0
  for res in results.values():
    complete = "yes" if not res.truncated else "**truncated**"
    rows.append(f"| {res.agent} | {HANDOFFS[res.agent]} | {res.attempts} | {complete} | "
                f"{', '.join(res.missing) or '-'} | {', '.join(res.extra) or '-'} | "
                f"{res.in_tokens:,} | {res.out_tokens:,} |")
    tot_in += res.in_tokens; tot_out += res.out_tokens
  rows.append(f"| **Total** | | | | | | **{tot_in:,}** | **{tot_out:,}** |")
  passed = sum(r.ok for r in results.values())
  return "\n".join(rows) + f"\n\n**{passed}/{len(results)} documents passed validation.**"

report = validation_report(results)
display(Markdown(report))

| Agent | Received | Attempts | Complete | Missing sections | Extra sections | In tok | Out tok |
|---|---|---|---|---|---|---|---|
| Product Manager | brief | 1 | yes | - | - | 359 | 2,715 |
| Designer | brief + PRD | 1 | yes | - | - | 2,766 | 3,160 |
| Engineer | brief + PRD | 2 | yes | - | - | 5,610 | 15,597 |
| Project Lead | brief + PRD + Design + Tech design | 2 | yes | - | - | 21,326 | 8,589 |
| QA Reviewer | brief + PRD + Design + Tech design + Plan | 3 | **truncated** | Top 3 Fixes | - | 40,962 | 31,000 |
| PM (closing) | brief + PRD + QA review | 2 | yes | - | - | 11,712 | 9,557 |
| **Total** | | | | | | **82,735** | **70,618** |

**5/6 documents passed validation.**

Notice how **input tokens grow** for later agents. Passing whole documents downstream is the price of giving an agent context.

## Read the documents

In [11]:
for label, res in results.items():
  display(Markdown(f"---\n# {label}  \n*written by {res.agent}*\n\n{res.text}"))

---
# PRD  
*written by Product Manager*

## Goals & Success Metrics

**Goal:** Improve retention of the web-based habit tracking app by adding a gamification layer (Badges, Streaks, Rewards) that reinforces daily check-ins.

| Metric | Baseline | Target (90 days post-launch) |
|---|---|---|
| D30 retention | Measure at launch | +10% relative lift vs. control |
| Weekly active users completing ≥3 check-ins/week | Measure at launch | +15% relative lift |
| % of active users with a streak ≥7 days | 0% | ≥30% |
| % of users earning ≥1 badge in first 7 days | 0% | ≥60% |
| Reward redemption rate (among users who earn rewards) | N/A | ≥40% |

**Guardrails:** No increase in habit-deletion rate. Page load time for the dashboard stays within 10% of baseline. Support tickets about streak errors stay under 1% of WAU.

**Validation:** A/B test, 50/50 split, for 4 weeks before full rollout.

## User Stories

1. **As a new user, I want to earn a badge after my first completed habit so that I feel early progress and stay motivated.**
   - AC1: Completing the first check-in triggers a "First Step" badge animation within 1 second.
   - AC2: The badge appears on my profile with its earned date.

2. **As a daily user, I want to see my current streak for each habit so that I'm motivated not to break it.**
   - AC1: Each habit card shows its current streak count in days, updated immediately after check-in.
   - AC2: The streak increments once per calendar day in my local time zone, regardless of how many check-ins occur that day.

3. **As a busy user, I want a limited number of streak freezes so that one missed day doesn't erase weeks of progress.**
   - AC1: I can apply a freeze to a missed day within 24 hours, and the streak remains unbroken.
   - AC2: The freeze count is visible and capped at 2 held at a time. I get a confirmation prompt before using one.

4. **As a user with a long streak, I want to unlock milestone badges (7, 30, 100 days) so that sustained effort feels recognized.**
   - AC1: Badges unlock automatically when the streak reaches each milestone, with an in-app notification.
   - AC2: Locked badges display on my badge page with progress toward unlocking.

5. **As a motivated user, I want to earn points that I can redeem for rewards so that consistency has tangible value.**
   - AC1: Each check-in awards points, and my balance is visible in the header and updates in real time.
   - AC2: I can redeem points from the rewards catalog, and my balance decrements accurately with a redemption confirmation.

6. **As a user who lapsed, I want a gentle prompt to restart so that a broken streak doesn't make me quit.**
   - AC1: After a streak breaks, I see a message showing my previous best streak and a "Start again" CTA.
   - AC2: My longest streak is preserved on my profile after a reset.

7. **As a privacy-conscious user, I want to control whether my badges and streaks are shared so that I can use the feature on my own terms.**
   - AC1: Settings include a toggle to hide badges and streaks from any shared or public view.
   - AC2: Users can turn gamification notifications off without losing progress.

8. **As a product team member, I want gamification events tracked so that we can measure impact on retention.**
   - AC1: Events (badge_earned, streak_incremented, streak_broken, freeze_used, reward_redeemed) are logged with user ID and timestamp.
   - AC2: A dashboard shows the metrics above, segmented by experiment cohort.

## Feature Requirements

### Streaks
- **MUST** calculate streaks per habit, based on the user's local time zone and the habit's schedule (daily or specific days).
- **MUST** display the current and longest streak on each habit card and the profile.
- **MUST** reset the streak to 0 after a missed scheduled day, unless a freeze is applied.
- **SHOULD** provide streak freezes (earn 1 per 14-day streak, hold max 2).
- **SHOULD** send an optional reminder when a streak is at risk (default: 8 pm local time).
- **COULD** offer a weekly-goal streak mode for habits that aren't daily.

### Badges
- **MUST** include a launch set of ≥12 badges: first check-in, streak milestones (7/30/100), total check-ins (10/50/250), and a "comeback" badge.
- **MUST** show earned and locked badges on a badge page, with unlock criteria and progress.
- **MUST** show a celebratory animation and notification on unlock.
- **SHOULD** make badge definitions config-driven so new badges can be added without a code release.
- **COULD** offer a shareable badge image for social platforms.
- **COULD** include seasonal or limited-time badges.

### Rewards
- **MUST** award points per check-in (10 points base, with a streak multiplier capped at 2x) and for badge unlocks.
- **MUST** maintain a ledger of points earned and spent, with the balance visible in the header.
- **MUST** provide a rewards catalog of digital items (themes, avatar frames, custom habit icons) redeemable with points.
- **SHOULD** apply server-side validation to prevent point duplication or manipulation from repeated check-in toggling.
- **COULD** add partner or physical rewards after launch, pending business-development review.

### Design Considerations
- **MUST** meet WCAG 2.1 AA, including a non-color-dependent streak indicator and reduced-motion support for animations.
- **MUST** be responsive across desktop and mobile web.
- **SHOULD** use positive framing for broken streaks (show the best streak, avoid shaming copy).
- **SHOULD** show a first-time tutorial or tooltip for gamification features.
- **COULD** offer a user-selectable "minimal mode" with reduced visual effects.

### Development Plan
- **MUST** deliver in phases:
  - Phase 1 (Weeks 1–4): event tracking, streak engine, streak UI, experiment framework.
  - Phase 2 (Weeks 5–8): badge system, badge page, notifications.
  - Phase 3 (Weeks 9–12): points ledger, rewards catalog, redemption flow.
  - Phase 4 (Weeks 13–16): A/B test, iteration, full rollout.
- **MUST** gate all features behind a feature flag with a kill switch.
- **MUST** backfill existing users' streaks and badges from historical check-in data at launch.
- **SHOULD** complete QA on time-zone and daylight-saving edge cases before the A/B test.

## Out of Scope
- Social features: leaderboards, friend challenges, and public feeds
- Monetary rewards, real-world prizes, and partner redemptions at launch
- Paid or premium purchase of points, freezes, or badges
- Native iOS/Android apps and push notifications (web and email only)
- AI-personalized challenges or dynamic difficulty
- Team or group habit tracking
- Retroactive editing of past check-ins to repair streaks beyond the freeze mechanic

---
# Design  
*written by Designer*

## UX Principles

- **Habit first, game second.** The check-in is the primary action. Gamification elements (streak, points, badges) are secondary and must never slow down or crowd the check-in flow. Check-in stays one tap from the dashboard.
- **Celebrate, don't interrupt.** Feedback is brief and non-blocking: inline animation and a dismissible toast. No modal that requires dismissal after a routine check-in. Reserve fuller celebrations for milestones (first badge, 7/30/100 days).
- **Immediate, legible feedback.** Streak count, points balance, and badge progress update in place at check-in, with the cause-and-effect visible (e.g., "+10 pts, 2x streak bonus").
- **Show progress, not just outcomes.** Locked badges show criteria and a progress bar ("4 of 7 days"). Streak cards show the next milestone.
- **Forgiveness by default.** Failure states (broken streak, missed day) lead with what the user has achieved (longest streak, total check-ins) and offer a clear next step. Copy is neutral and encouraging.
- **Transparency of rules.** Every rule is discoverable in context: how points are calculated, how freezes are earned and used, when a day rolls over (local midnight). Show the time zone in use.
- **User control.** Users can reduce, hide, or turn off gamification elements without losing progress.
- **Progressive disclosure.** New users see streaks and the first badge only. Points, rewards, and freezes are introduced as they become relevant, via a short dismissible tooltip tour.

## Key Screens

- **Dashboard (habit list):** Check in per habit via one tap, with the ability to undo. Each habit card shows current streak, longest streak, and a freeze indicator when a day is at risk. The header shows the points balance and a freeze count (x/2). The user sees check-in feedback (points earned, streak increment) inline.
- **Check-in celebration (toast/inline animation):** Brief confirmation of points and streak. Milestone and badge unlocks expand into a richer, still dismissible, card with a "View badge" link.
- **Habit detail:** Review streak history on a calendar, with check-in days, freeze-protected days, and missed days each marked by distinct icons and labels (not color alone). Shows current and longest streak and the next milestone.
- **Streak freeze prompt:** Appears when a scheduled day was missed and a freeze is available within the 24-hour window. States the cost ("Use 1 of 2 freezes to protect your 18-day streak?"), the remaining freezes afterward, and the deadline. Primary action: Use freeze. Secondary: Let it go. Confirmation is required, as the PRD specifies.
- **Streak broken / restart screen:** Shows previous best streak, total check-ins, and a "Start again" CTA. Optionally shows the comeback badge criteria. Copy is supportive, with no guilt language.
- **Badge page:** Grid of earned and locked badges. Earned badges show the date; locked badges show criteria and progress. Filter by All, Earned, and In progress. Each badge has a detail view.
- **Rewards catalog:** Browse digital items with their point costs. Items the user can't yet afford show how many points remain. Redeeming opens a confirmation dialog with the resulting balance, and a success state lets the user apply the item immediately.
- **Points history (ledger):** A chronological list of points earned and spent, with the reason for each entry, so the balance is verifiable.
- **Profile:** Displays earned badges with dates, longest streaks, and the balance. Visibility respects the privacy setting.
- **Settings > Gamification:** Toggles for hiding badges/streaks from shared or public views, gamification notifications, the 8 pm at-risk reminder (with time editing), and minimal mode (reduced effects). Copy confirms that turning anything off does not erase progress.
- **First-time tutorial:** A 3–4 step tooltip sequence (streaks, badges, then points/freezes as unlocked). Skippable and re-openable from Help.

## Motivation vs. Manipulation

| Risk | Where it could become a dark pattern | Design response |
|---|---|---|
| **Loss aversion / streak anxiety** | Streak counters, "at risk" reminders, and a reset to 0 can create compulsive, fear-driven use, or lead users to check in without really doing the habit. | Reminders are opt-in/opt-out, calm in tone, and sent once. Never use countdown timers, red alarms, or "you'll lose everything" copy. The longest streak is always preserved and shown. Freezes are framed as a planned tool, not a rescue offer. |
| **Shame on lapse** | Sad or guilt-laden messaging after a break. | Lead with achievements and a "Start again" CTA. The comeback badge rewards returning. No negative iconography. |
| **Gaming the system** | Rewarding quantity can push users to log false check-ins or toggle repeatedly for points. | Points are validated server-side. Undo reverses points transparently. Habits stay user-defined, and nothing in the UI nudges users to create trivial habits for points. |
| **Variable or hidden rewards** | Surprise rewards and opaque point formulas resemble slot-machine mechanics. | All point values, multipliers (cap 2x), and badge criteria are published. No randomized rewards or loot boxes. |
| **Sunk-cost lock-in** | Large streaks and balances can make users feel unable to leave or pause a habit. | Allow pausing, archiving, or deleting habits without penalty or guilt dialogs. Monitor the habit-deletion guardrail. Let users export their data. |
| **Fake urgency and spending pressure** | Pushing users to spend points quickly, scarce-item pressure, or seasonal "FOMO" badges. | Points don't expire. No countdown timers on the catalog. Seasonal badges, if built, are clearly labeled, and a missed one never penalizes progress. |
| **Confirmshaming and friction on opt-out** | Making gamification hard to turn off. | Settings have symmetric, neutral toggles. Declining the tutorial or reminders carries no penalty or guilt copy. |
| **Notification overload** | Excess prompts to drive retention metrics. | Cap gamification notifications, with one at-risk reminder per day max. Users can mute them without affecting their progress. |
| **Extrinsic over intrinsic motivation** | Rewards overshadow the user's own reasons for the habit. | Keep the user's own habit goal and notes prominent on the cards. Rewards are cosmetic only. Celebrate progress toward the user's goals, not just points. |

Design review checkpoint: before the A/B test, review all copy and flows against this table. Metrics beyond retention (habit-deletion rate, opt-out rate, support tickets) are monitored as signals of manipulation.

## Accessibility

- **WCAG 2.1 AA** across all new screens: contrast ≥4.5:1 for text and ≥3:1 for UI components and graphics, including locked-badge states, where reduced visual weight must not fall below the threshold.
- **Not color alone:** Streak status (active, at risk, frozen, broken) uses distinct icons, text labels, and shapes. Calendar days use different patterns/symbols for completed, frozen, and missed.
- **Reduced motion:** Honor `prefers-reduced-motion` by replacing confetti and badge animations with a static or simple fade state that conveys the same information. Minimal mode provides a manual override. Nothing flashes more than three times per second.
- **Keyboard and focus:** All check-ins, freeze prompts, redemption dialogs, and tooltips are fully operable by keyboard with visible focus indicators. Modals trap focus and return it on close. Tutorial tooltips are dismissible with Esc.
- **Screen readers:** Announce dynamic updates through polite `aria-live` regions (e.g., "Meditation checked in. Streak is now 8 days. 10 points earned."). Do not make a toast the only means of conveying information. Badges have text alternatives with name, status, earned date, and progress. Progress bars use `role="progressbar"` with value text. The points balance in the header announces changes without stealing focus.
- **Timing:** Toasts stay at least 6 seconds, pause on hover and focus, and are available later in an in-app notification list. The freeze window (24 hours) is stated in plain text, with no short time-limited interactions.
- **Targets and responsiveness:** Touch targets ≥44x44 px. Layouts reflow at 320px width and support 200% zoom and text resizing without loss of content. Check-in buttons have visible labels, not icon-only.
- **Clear language:** Plain-language copy for rules (points, freezes, time zones). Avoid jargon and ambiguous icons, and provide text labels beside all icons.
- **Cognitive accessibility:** Minimal mode and notification toggles reduce stimulation. The tutorial is skippable and re-accessible. Confirmation dialogs for freezes and redemptions prevent accidental actions, and the user can undo check-ins.
- **Testing:** Run automated checks (axe) plus manual testing with NVDA/VoiceOver, keyboard-only navigation, and zoom. Include users with disabilities in usability testing before the A/B launch.

---
# Technical design  
*written by Engineer*

## Data Model

Existing tables `users`, `habits` and `check_ins` are extended. New tables are marked **(new)**. All timestamps are `timestamptz` in UTC. `local_date` is a `date` in the user's time zone.

| Table | Key fields |
|---|---|
| `users` (ext.) | `tz` (IANA string), `cohort` (control/treatment, from experiment framework) |
| `gamification_settings` **(new)** | `user_id` PK, `hide_public` bool, `notifications_enabled` bool (in-app and email), `reminder_enabled` bool, `reminder_time` default 20:00 local |
| `habits` (ext.) | `schedule_type` (daily/days), `schedule_days` (7-bit mask), `schedule_effective_from` date, `created_at`, `deleted_at`, `icon_id` (nullable, FK to reward) |
| `check_ins` (ext.) | `id`, `habit_id`, `user_id`, `local_date`, `tz_used`, `checked_at`, `voided_at` (nullable, set on undo), `source` (user/backfill). Unique `(habit_id, local_date) WHERE voided_at IS NULL` |
| `streaks` **(new)** | `habit_id` PK, `current_count`, `longest_count`, `last_counted_date`, `status` (active/pending_break/broken), `broken_streak_count` (streak value at last break, used for the restart prompt), `next_eval_at` (UTC, indexed) |
| `freeze_balances` **(new)** | `user_id` PK, `balance` (CHECK 0..2) |
| `freeze_events` **(new)** | `id`, `user_id`, `habit_id`, `type` (earned/used), `missed_date` (for used), `created_at`. Unique `(habit_id, missed_date) WHERE type='used'` |
| `badge_definitions` **(new)** | `id`, `key`, `name`, `description`, `criteria_type` (first_checkin/streak_milestone/total_checkins/comeback), `threshold`, `points_reward`, `active`. At least 12 rows at launch, loaded via config/seed rather than code. |
| `user_badges` **(new)** | `user_id`, `badge_id`, `earned_at`, `habit_id` (nullable), `source` (live/backfill). PK `(user_id, badge_id)` |
| `points_ledger` **(new)** | `id`, `user_id`, `delta`, `reason` (checkin/badge/redemption), `ref_type`, `ref_id`, `idempotency_key` UNIQUE, `created_at` |
| `point_balances` **(new)** | `user_id` PK, `balance` (CHECK ≥ 0), `version`. Cache of the ledger sum, updated in the same transaction as each ledger insert. |
| `rewards_catalog` **(new)** | `id`, `type` (theme/avatar_frame/habit_icon), `name`, `cost`, `active` |
| `redemptions` **(new)** | `id`, `user_id`, `reward_id`, `ledger_id`, `created_at` |
| `notifications` **(new)** | `id`, `user_id`, `type`, `payload` jsonb, `created_at`, `read_at` |
| `events` **(new)** | `id`, `user_id`, `name` (badge_earned, streak_incremented, streak_broken, freeze_used, reward_redeemed), `props` jsonb, `cohort`, `source` (live/backfill), `occurred_at`. Written via transactional outbox. |

Source of truth is `check_ins` plus `freeze_events`. `streaks` is a recomputable cache.

## Streak Logic

**Core rules**
- Streaks are per habit and count scheduled days only. A scheduled day is "met" if a non-voided check-in exists for that `local_date`, or a freeze was applied to it.
- Day N+1 extends the streak only if the previous scheduled day was met.
- Multiple check-ins on one day collapse to one, enforced by the unique index. The streak increments at most once per `local_date`.
- A check-in on a non-scheduled day is recorded and earns points but does not change the streak (assumption, see Technical Risks).
- A frozen day bridges the streak but does not increment it.
- `longest_count = max(longest_count, current_count)`. It is never decremented, including after a reset.
- Days before `habit.created_at` or `schedule_effective_from` are never "missed". Schedule changes apply forward only.

**Computation**
- A pure function `computeStreak(checkIns, freezes, schedule, tz, now)` is the single implementation. It is used by the live path, the nightly job, the backfill and repair recompute.
- Live path: one transaction per check-in.
  - Lock the `streaks` row (`SELECT … FOR UPDATE`).
  - Insert the check-in.
  - Update the streak.
  - Evaluate badges and freeze earning.
  - Insert ledger rows and the notification.
  - Insert the outbox events.
- Freeze earning: grant 1 when `current_count` reaches a multiple of 14 and `balance < 2`.
- A scheduler picks `streaks WHERE next_eval_at <= now()` to finalize breaks, so there is no per-minute scan of all users.
- Reads also lazily evaluate the streak, so the UI is correct even if the job is delayed.

**Missed day and freeze window**
- Missed scheduled day D ends at local midnight. The streak enters `pending_break` and the freeze window runs until local midnight at the end of D+1 (calendar-based "24 hours").
- If the user holds 0 freezes, the break is finalized immediately at the end of D.
- A freeze applied in the window recomputes the streak. A check-in on D+1 made before the freeze still joins the restored streak.
- Break finalization sets `current_count = 0` and `status = broken`, stores `broken_streak_count`, and emits `streak_broken`. The restart message reads `broken_streak_count` and `longest_count`.
- A freeze is rejected if D is not a missed scheduled day, the window has closed, the balance is 0, or D is already frozen. Freezes are idempotent per `(habit, date)`.

**Time zones and DST**
- `local_date` is always derived server-side from `checked_at` and the user's IANA `tz`. The client never sends a date.
- Date arithmetic uses calendar dates, never `+24h`. 23-hour and 25-hour DST days therefore count as one day.
- `next_eval_at` is recomputed from the tz library for each user.
- Time zone change (travel):
  - Existing `local_date` values are immutable.
  - Break evaluation uses the tz in effect at evaluation time.
  - If the new tz yields a `local_date` that already has a check-in, no second check-in is allowed.
  - An eastward move can skip a calendar day. Handling is in Technical Risks.

**Undo and toggling**
- Undo is allowed only for the current `local_date`. It sets `voided_at` and recomputes the streak, so the increment is reversed.
- Points are awarded once per `(habit, local_date)` via `idempotency_key = checkin:{habit_id}:{local_date}`.
- Re-checking after an undo does not re-award points, and undo does not claw them back. This prevents duplication and negative balances.
- Badges and badge points are permanent and are not revoked by undo.

**Backfilled check-ins**
- Users cannot create past-dated check-ins, since retroactive repair is out of scope. The only past-dating path is the launch migration.
- Launch backfill runs `computeStreak` over historical `check_ins` in batches, in a resumable job.
  - It uses the user's current `tz`, which is an approximation.
  - It writes `streaks` and `user_badges` with `source='backfill'`.
  - It suppresses animations and notifications and tags events `source='backfill'` so metrics exclude them.
  - It does not grant freezes or points (default, see Technical Risks).
- Late-synced requests: `local_date` is set from server receipt time, so a delayed client cannot backdate.

## API Endpoints

All endpoints are auth-required and checked against the feature flag, cohort and kill switch. When the flag is off they return 404 for gamification routes, and core check-in still works. Streaks are derived from `check_ins`, so they are correct on re-enable. Mutating endpoints accept an `Idempotency-Key` header.

| Method & Path | Purpose |
|---|---|
| `POST /api/habits/{id}/check-ins` | Check in for the current local day. Returns `{checkIn, streak, pointsAwarded, balance, badgesUnlocked[], freezeEarned}`. |
| `DELETE /api/habits/{id}/check-ins/today` | Undo today's check-in. |
| `GET /api/habits` (ext.) | Adds `streak: {current, longest, status, atRisk}` per habit in a single join on `streaks`. |
| `GET /api/me/gamification` | Balance, freeze count, longest streaks, badge count, flag and cohort, and a restart-prompt payload if a streak is broken. |
| `GET /api/freezes` | Balance plus eligible missed days and window deadlines. |
| `POST /api/habits/{id}/freezes` `{date}` | Apply a freeze. The confirmation prompt is client-side. |
| `GET /api/badges` | All definitions with `earned_at`, `progress {current, target}`, and locked/earned state. |
| `GET /api/rewards` | Catalog with affordability. |
| `POST /api/rewards/{id}/redemptions` | Atomic redemption: lock `point_balances`, check balance, insert ledger, redemption and event. Returns the new balance. |
| `GET /api/points/ledger?cursor=` | Paginated earn and spend history. |
| `GET/PATCH /api/me/gamification-settings` | `hide_public`, `notifications_enabled`, `reminder_enabled`. |
| `GET /api/notifications`, `POST /api/notifications/{id}/read` | In-app unlock and streak notifications. |
| `GET /api/users/{id}/profile` (ext.) | Existing shared view. Server omits badges and streaks when `hide_public` is true. |
| `GET /api/admin/gamification-metrics?cohort=&from=&to=` | Internal metrics dashboard feed (PRD metrics by cohort). Role-restricted. |

Background jobs (not public endpoints):
- Streak evaluator, driven by `next_eval_at`.
- Email reminder sender, which runs at the user's local `reminder_time` for at-risk streaks and respects the settings.
- Backfill migration.
- Outbox publisher.

## Technical Risks

- **Time zone and DST correctness:** This is the main source of "streak error" tickets, which have a guardrail of under 1% of WAU. Mitigations:
  - Use a single pure function and property tests across DST transitions and UTC±14.
  - Run QA before the A/B test, as the PRD requires.
  - Keep a repair-recompute admin tool.
- **Time zone manipulation:** A user changing their tz could gain an extra day or a duplicate check-in. Mitigation: validate the IANA tz, throttle tz changes, and keep the unique `(habit, local_date)` index. Eastward travel can skip a day and look like a missed day. Whether to give a grace rule is an open question.
- **Concurrency and abuse:** Rapid toggling, double submits and multi-tab use are handled by row locks, unique ledger keys, idempotency keys and `CHECK (balance >= 0)`.
- **Dashboard performance:** The dashboard must stay within 10% of baseline load time. Mitigation: denormalized `streaks` and `point_balances`, no per-request recompute on the happy path, and lazy evaluation only when `next_eval_at` has passed.
- **Backfill:** Large batch writes can load the database, and historical tz is unknown. Mitigation: throttled, resumable, idempotent batches, run before the flag is enabled.
- **Config-driven badges:** New badges work only for existing `criteria_type` evaluators. A new criteria type needs a code release.
- **"Real-time" balance:** Delivered via the check-in response and a refetch on focus. Cross-tab or push-style updates would need polling or SSE.
- **Experiment integrity:** If control users do not run streak computation, it cannot be compared later or backfilled cleanly. Mitigation: compute events silently for control users, flagged `source`/`cohort`.

**Open questions (not specified in the PRD; defaults assumed above):**
1. Are freezes held per user or per habit? Assumed per user, with the earn trigger on any habit's 14-day multiple.
2. If a freeze is earned at the cap of 2, is it forfeited or queued? Assumed forfeited.
3. Does a check-in on a non-scheduled day count toward the streak? Assumed no, points only.
4. What is the streak multiplier formula up to the 2x cap, and are points awarded for backfilled badges? Assumed a config-driven formula and no backfill points.
5. What is the exact "comeback" badge criterion? Proposed: a new streak reaching N days after a break of a prior streak, with N in config.
6. Are retroactive freezes or points granted to existing users based on backfilled streaks? Assumed no, effective from launch.
7. Are rewards single-redemption per user? Assumed yes for themes, frames and icons, which would need a unique `(user_id, reward_id)` constraint.
8. Is undo limited to the same local day? Assumed yes.
9. Is the freeze window literally 24 hours or until local midnight of the next day? Assumed the latter, which differs by at most 1 hour at a DST change.
10. Should the weekly-goal streak mode (COULD) be deferred? Assumed deferred, with the schema not blocking it.

---
# Plan  
*written by Project Lead*

## Milestones

| Milestone | Scope | Duration | Depends on |
|---|---|---|---|
| M0. Decisions & kickoff | Confirm the defaults for the 10 technical-design open questions with PM and design: freezes per user, forfeit at cap, non-scheduled-day behavior, multiplier formula, comeback criterion, no retro freezes or points, single redemption, same-day undo, freeze window to next local midnight, weekly-goal mode deferred. Finalize the tracking plan. | Week 1 | None |
| M1. Foundations | Schema migrations (streaks, freezes, settings, events, outbox) and the `check_ins` extension. Feature flag with kill switch (flag off returns 404 on gamification routes, core check-in unaffected). Experiment framework with 50/50 cohort assignment. Event logging via transactional outbox, including silent computation for control users. | Weeks 1–3 | M0 |
| M2. Streak engine & UI | `computeStreak` pure function with property tests (DST, UTC±14). Live check-in transaction, undo, `next_eval_at` evaluator, lazy read evaluation. Freeze earn/apply/reject rules. Habit card streak display with non-color indicator, habit detail calendar, freeze prompt, restart screen. Header freeze count. Check-in and freeze APIs. | Weeks 2–4 | M1 |
| **Phase 1 exit** | Streaks, freezes and events work end to end behind the flag; streak logic unit and property tests pass. | Week 4 | M1, M2 |
| M3. Badge system | Badge definitions seeded from config (≥12 badges, with the four criteria evaluators). Badge evaluation in the live path, `user_badges`, badge points hook (credited once M5 ships). `GET /api/badges` with progress. | Weeks 5–6 | Phase 1 exit |
| M4. Badge UI, notifications & settings | Badge page (filters, detail view, progress bars). Celebration toast and card with reduced-motion support. In-app notifications list. Settings > Gamification (hide public, notifications, reminder with time, minimal mode). Profile privacy enforcement. Email at-risk reminder job. Tutorial steps for streaks and badges. | Weeks 6–8 | M3 |
| **Phase 2 exit** | Badges and notifications complete; the privacy toggle is verified server-side. | Week 8 | M3, M4 |
| M5. Points & rewards | Points ledger, `point_balances`, idempotent check-in points with the capped multiplier. Rewards catalog, atomic redemption, ledger history screen, header balance. Tutorial steps for points and freezes. | Weeks 9–11 | Phase 2 exit |
| M6. Backfill | Resumable, throttled, idempotent backfill job. It writes streaks and badges tagged `source=backfill`, with no animations or notifications and no freezes or points. Staging dry run, validation sample, production run with the flag still off. | Build weeks 9–10; rehearse week 11; production run week 12 | Phase 2 exit; production run also needs M5 complete |
| M7. Metrics dashboard | Admin metrics endpoint and dashboard for PRD metrics and guardrails by cohort. Baseline capture for D30 retention and ≥3 check-ins/week. | Weeks 10–12 | M1; M5 for reward events |
| M8. Pre-A/B hardening | Time-zone and DST QA (including tz change and eastward-travel cases). Accessibility audit (axe, NVDA/VoiceOver, keyboard, 200% zoom) and usability testing with users with disabilities. Design review of all copy and flows against the Motivation vs. Manipulation table. Dashboard load-time check (within 10% of baseline). Concurrency and toggle-abuse tests. Kill-switch drill. | Weeks 11–12 | M2, M4, M5 (M6 in parallel) |
| **Go/no-go for A/B** | M6 backfill complete, M7 dashboard live, M8 exit criteria met. | End of week 12 | M6, M7, M8 |
| M9. A/B test | Enable flag for 50/50 split for 4 weeks. Monitor primary metrics and guardrails (habit deletion, load time, streak-error tickets, opt-out rate) weekly. Only bug and copy fixes during the test. | Weeks 13–16 | Go/no-go |
| M10. Iteration & rollout | Analyze results, apply fixes, make the rollout decision, and stage the full rollout using the flag. | Week 16 (rollout decision at test end) | M9 |

## Team & Roles

| Role | Responsibility |
|---|---|
| Delivery/Project Lead | Plan, milestones, dependencies, risk log, go/no-go coordination, weekly status |
| Product Manager | Closes open questions in M0, owns the PRD, owns the A/B analysis and rollout decision |
| Product Designer (+ content) | Screens, copy, tutorial, minimal mode, pre-A/B design review against the dark-pattern table |
| Backend Engineers (2) | Streak engine, ledger and redemption, badges, APIs, jobs, backfill, outbox |
| Frontend Engineers (2) | Dashboard and habit card, calendar, prompts, badge page, rewards, settings, accessibility implementation |
| QA Engineer | Test plan, time-zone/DST suites, regression, backfill validation, accessibility manual testing |
| Data/Analytics Engineer | Event schema, experiment framework support, metrics dashboard, baselines, cohort analysis |
| DevOps/SRE (part-time) | Feature flag and kill switch, backfill runbook and DB load monitoring, performance monitoring |
| Support Lead (part-time) | Streak-error ticket tagging and triage, rollout readiness |

## Risks & Mitigations

| Risk | Mitigation |
|---|---|
| Time-zone/DST errors cause streak-error tickets (guardrail <1% of WAU) | Single `computeStreak` implementation, property tests across DST and UTC±14, dedicated QA in M8, repair-recompute admin tool, ticket tagging from day one |
| Time-zone manipulation or eastward travel skipping a day | Validate IANA tz, throttle tz changes, rely on the unique `(habit, local_date)` index. PM decision on a grace rule is needed in M0 or tracked as a known limitation. |
| Open questions unresolved (freeze scope, comeback criterion, multiplier formula) | Resolve in M0 with the documented defaults. Changes after Phase 1 trigger re-estimation. |
| 16-week schedule leaves no slack; the 4-week A/B fills all of Phase 4 | Hold the week-12 go/no-go strictly. Overlap the backfill, metrics and hardening work with M5. Limit in-test changes to fixes. If the go/no-go slips, the A/B start moves and rollout shifts with it. |
| Backfill load on the database, or approximate historical tz | Throttled, resumable, idempotent batches. Staging rehearsal. Run before the flag is enabled. Tag `source=backfill` and exclude from metrics. |
| Dashboard load time degrades by more than 10% | Denormalized `streaks` and `point_balances`, no happy-path recompute, performance test in M8 |
| Duplicate points from toggling, double submits or multi-tab use | Row locks, unique idempotency keys, `CHECK (balance >= 0)`, concurrency tests |
| Experiment integrity (control users uncomputed or contaminated) | Silent streak computation and events for control users with cohort and source tags. Verify cohort assignment before the A/B. |
| Accessibility gaps found late | Accessibility acceptance criteria in each story, axe in CI, manual audit and usability testing with users with disabilities in M8 before the A/B |
| Gamification drives manipulative behavior (habit deletion, opt-outs) | Design review against the Motivation vs. Manipulation table before the A/B. Monitor deletion rate, opt-out rate and tickets during the test. Use the kill switch if guardrails are breached. |
| New badge types need a code release | Launch badges use only the four existing criteria types. Anything else is post-launch. |
| "Real-time" balance only updates on check-in response and refetch on focus | Accept as the v1 behavior. Note cross-tab or push updates as a future enhancement. |
| Scope creep into COULD items (weekly-goal mode, shareable badges, seasonal badges, partner rewards) | Not scheduled. Only revisit after the Phase 4 results and if capacity remains. |

---
# QA review  
*written by QA Reviewer*

## Gaps

1. **Streak unit for non-daily habits.** The PRD counts streaks in "days". The Tech design counts scheduled days only. It is unspecified what "7/30/100 days", "4 of 7 days", the 14-day freeze trigger and the comeback N mean for a Mon/Wed/Fri habit.
2. **Badge catalog.**
   - The PRD requires ≥12 badges but names only 8: first check-in, 3 streak, 3 total check-ins and comeback. Four are undefined.
   - `points_reward` values are undefined.
   - Streak badges are PK `(user_id, badge_id)`, so they are once per user. The PRD and Design imply milestones per habit.
   - Whether "total check-ins" includes non-scheduled-day check-ins is unstated.
3. **Points and rewards economy.**
   - The streak multiplier formula (up to the 2x cap) is not given. The Tech design says only "config-driven".
   - Badge point values, catalog items, item counts and costs are undefined.
   - The ≥40% redemption target cannot be sized without costs.
4. **Applying rewards.**
   - Design says the success state lets the user "apply the item immediately". There is no data model or API for equipped themes or avatar frames. Only `habits.icon_id` exists.
   - It is unstated whether the app already has themes or avatars.
   - The single-redemption constraint (open question 7) is not in the schema.
5. **Time zone ownership.**
   - Where `users.tz` comes from (browser detection or user-set) is unspecified.
   - There is no endpoint to change it, and the throttle value is not given.
   - Design requires showing the zone in use, but no API field returns it.
   - The eastward-travel grace rule has no owner and no default.
6. **Freeze and event farming via undo.**
   - Undo and re-check at day 14 (or any multiple of 14) would re-trigger freeze earning. `freeze_events` has no uniqueness on `earned`.
   - `streak_incremented` events and day-7/30/100 badges are likewise unspecified on undo and redo. Badges are permanent but the events are not addressed.
7. **Freeze and break UX.**
   - With one freeze and several habits missed, the prompt's placement, trigger and ordering are undefined.
   - What the card shows during `pending_break` is undefined.
   - `atRisk` is never defined, yet it drives the card indicator and the reminder.
   - Forfeiting a freeze at the cap (open question 2) is not communicated in Design.
8. **Restart prompt.**
   - Dismissal state is not stored.
   - It is unclear whether it is shown per habit or aggregated.
   - "Start again" has no defined effect, since the streak is already 0.
   - Design wants "total check-ins", but that is not in the `/api/me/gamification` payload.
   - "Previous best" in the PRD could mean `broken_streak_count` or `longest_count`.
9. **Experiment design.**
   - Missing: assignment unit and timing (new versus existing users), sample size or MDE, significance thresholds and the rollout decision rule.
   - "Active user", "retention" and the load-time measurement method (RUM versus synthetic, page, percentile) are undefined.
   - It is unspecified whether backfilled and silent control data count toward any metric.
10. **Backfill.**
    - Unstated: whether milestone badges derive from the current or the longest historical streak.
    - Unstated: whether existing users with a 100-day backfilled streak get the 2x multiplier on day one.
    - Unstated: whether control users are backfilled, and what happens to them at rollout (a flood of silent-earned badges?).
    - The interaction of the week-12 backfill with live computation since week 1–3 is unspecified.
11. **Flag-off and kill switch behavior.**
    - Points are ledger-based and not recomputable. Check-ins made while the flag is off earn nothing.
    - Equipped cosmetics behavior is unspecified.
    - Response shapes for `POST check-ins` and `GET /api/habits` when the flag is off are unspecified.
    - Using the kill switch mid-test invalidates the experiment, and no policy is given.
12. **Notifications.**
    - Unspecified: whether the reminder is on by default, its channel (email only?), and whether it is per user or per habit.
    - Unspecified: email consent and unsubscribe handling, and the numeric notification "cap".
    - It is unclear whether `notifications_enabled=false` also suppresses inline celebrations, which the PRD makes a MUST.
13. **Design promises with no spec elsewhere.**
    - Pausing or archiving habits "without penalty" and data export appear only in Design. The effect on streaks, points and badges is undefined.
    - Habit schedule editing and deletion effects are also undefined.
14. **Non-functionals.**
    - No scale, latency SLO or browser support matrix.
    - No event retention or privacy and GDPR treatment.
    - Admin "role-restricted" roles are undefined.
    - The PRD refers to "shared or public views" while social features are out of scope. Which views exist is unclear.
15. **Event taxonomy.** Opt-out rate, which Design and Plan monitor, has no event. There is also no freeze-earned, tutorial or settings-change event, and no check-in event for the metrics.

## Contradictions

16. **Undo and points.** Design (Gaming the system): "Undo reverses points transparently." Tech design: undo never claws points back, and re-checking does not re-award them. The PRD only says server validation prevents duplication.
17. **Freeze window.** PRD US3 AC1 and Design say "24 hours" and "24-hour window", stated in plain text. Tech and Plan use calendar logic ending at local midnight of D+1. Open question 9 resolves this only in the Tech design and Plan M0, so the user-facing copy would be wrong or unclear.
18. **Freeze priority and rule.**
    - PRD marks freezes SHOULD, yet US3, the Design header and prompt, and Plan M2 treat them as core.
    - The PRD's "1 per 14-day streak" is ambiguous. Tech implements "every multiple of 14 on any habit" into a per-user pool.
19. **"Real time" balance.** PRD US5 AC1 says the balance "updates in real time". Tech and Plan accept refetch-on-focus only, so there is no cross-tab update.
20. **Settings.**
    - Design's UX principle says users can "turn off" gamification, but its Settings screen has no master toggle.
    - Design Settings includes reminder time editing and minimal mode. The Tech `gamification_settings` table and PATCH endpoint lack both.
    - Minimal mode is a PRD COULD but is scheduled in Plan M4.
21. **Progressive disclosure versus header.** Design says new users see only streaks and the first badge. The same document's Dashboard, and PRD US5, put points and the freeze count in the header from the start. Plan M2 ships the header freeze count in Phase 1, but the freeze and points tutorial is in M5.
22. **Badge points.** PRD makes points for badge unlocks a MUST. Plan M3 defers the "hook" until M5. Tech gives no backfill points. Badges earned in weeks 5–11, and backfilled ones, would never get points.
23. **Timeline.**
    - The PRD's Phase 4 includes A/B, iteration and full rollout in weeks 13–16. Plan M9 fills all four weeks, M10 is "week 16", and the PRD allows "bug fixes only".
    - A 4-week test cannot measure D30 for late enrollees, yet PRD targets are "90 days post-launch".
    - Plan M2 (weeks 2–4) depends on M1 (weeks 1–3). M1 depends on M0, which is also in week 1, so schema such as per-user `freeze_balances` is built before the decisions.
24. **Baselines.** PRD sets baselines of 0% for "streak ≥7" and "badge in first 7 days". Backfill gives existing users streaks and badges, and control users silently compute them. The Tech design excludes `source=backfill` from events only.
25. **Plan versus PRD.**
    - Plan Risks claims "accessibility acceptance criteria in each story", but the PRD stories contain none.
    - The PRD makes DST QA a SHOULD, while the Plan gates go/no-go on it.
    - The Plan's go/no-go on "M8 exit criteria" is undefined.
26. **Eastward-travel rule.** Plan Risks says the PM decides it in M0. The M0 scope and the Tech design's 10 open questions both omit it.

## Untestable Requirements

27. US1 AC1 "within 1 second" has no start point, percentile, network or device condition.
28. US5 AC1 "real time" and US2 AC1 "immediately" have no latency bound (see item 19).
29. The PRD's "+10% relative lift" and "+15% relative lift" have no significance level, power, sample size or definition of "active" or "retention" (see item 9).
30. The guard

---
# Closing  
*written by PM (closing)*

## Executive Summary
- QA found real gaps in the PRD: undefined economy values, streak units for non-daily habits, experiment design, backfill rules and flag-off behavior. I accept 29 of the 30 items, and the PRD will be revised before M0 closes.
- Freezes move from SHOULD to MUST, with one defined earning rule. Points are never clawed back on undo. All events and rewards become idempotent, which closes the undo-farming loophole.
- The A/B test will use a defined design: assignment unit, MDE, significance thresholds, metric definitions and a decision rule. Backfilled data is excluded from baselines. D30 gets a longer read via a persistent holdout.
- Scope is trimmed for launch. Minimal mode, data export and cross-tab real-time updates are deferred, and the "shared/public view" language is removed because social features are out of scope.
- Item 30 was cut off in the QA text I received. The Tech design, Design and Plan documents were not in my input, so those claims are unverified.

## Decisions in Response to QA
1. ACCEPT: "streak" will count consecutive *scheduled* occurrences, and all thresholds (7/30/100, 14-day freeze, comeback N) use that unit. The UI will say "7 check-ins in a row" for non-daily habits.
2. ACCEPT: I will name all 12+ badges and set `points_reward` values. Milestone badges are once per user (first habit to reach it). Total check-ins count one per habit per day, including non-scheduled days.
3. ACCEPT: I will publish the multiplier formula, badge points, catalog items and costs. The ≥40% redemption target is provisional until costs are set.
4. ACCEPT: I will add an equipped-items model and API, and confirm which themes and avatars already exist. The launch catalog shrinks to what is buildable. Single-redemption items get a schema constraint.
5. ACCEPT: Time zone is browser-detected with a user override, throttled (value set in M0). The API will return the zone in use. The eastward-travel grace rule is owned by me, with a default set in M0.
6. ACCEPT: Freeze earning and `streak_incremented` events become idempotent per (user, habit, milestone/date). Undo and redo cannot re-trigger them.
7. ACCEPT: I will define `atRisk` and the `pending_break` card state. When several habits are missed, one prompt lists them in streak-length order. Design will communicate forfeiting a freeze at the cap.
8. ACCEPT: The restart prompt is per habit, shows `longest_count`, and its dismissal is stored. "Start again" becomes a dismiss-and-focus action, and total check-ins is added to the payload.
9. ACCEPT: I will specify the assignment unit and timing, MDE and sample size, significance thresholds, the rollout decision rule, and definitions of active user, retention and the load-time measurement method (RUM, p75).
10. ACCEPT: Backfill derives milestone badges from the longest historical streak. The multiplier uses the current streak only. Control users are not backfilled visibly, and at rollout badges are granted quietly with one summary message. Backfill runs once, with a defined cutover from live computation.
11. ACCEPT: Check-ins made while the flag is off earn nothing, and I will state this plainly. Cosmetics stay equipped but are read-only, and flag-off response shapes will be specified. The kill switch is for safety only and restarts the experiment.
12. ACCEPT: The reminder is opt-in, email only, per user. I will set a numeric cap, add consent and unsubscribe handling, and let `notifications_enabled=false` suppress inline celebrations too, with a MUST exception for accessibility.
13. ACCEPT (data export DEFER): Pause, archive, edit-schedule and delete effects on streaks, points and badges will be specified. Data export is deferred to post-launch and removed from Design.
14. ACCEPT: I will add scale, latency SLO, browser matrix, event retention and GDPR treatment, and define the admin roles. US7 is reworded to profile visibility only, since no shared or public views exist.
15. ACCEPT: I will add events for opt-out, freeze_earned, tutorial and settings changes, plus a check-in event for the metrics.
16. ACCEPT: Undo never claws back points and re-checking does not re-award them. Design copy is corrected to match.
17. ACCEPT: User-facing copy will say "until the end of the next day (your local time)" instead of "24 hours".
18. ACCEPT: Freezes become MUST. The rule is one freeze earned each time any habit hits a multiple of 14, once per habit per milestone, into a per-user pool capped at 2.
19. ACCEPT: US5 is reworded to "updates within 2s of action and on tab focus". There is no cross-tab sync at launch.
20. ACCEPT (minimal mode DEFER): I will add a master gamification toggle and reminder-time editing to the settings table and API. Minimal mode moves post-launch and out of Plan M4.
21. ACCEPT: Points and the freeze count appear in the header only after first earned. The tutorial moves earlier to ship with the first Phase 1 element it explains.
22. ACCEPT: Badge points are awarded at unlock from Phase 2, so the hook is not deferred to M5. Backfilled badges grant their badge points once, with no per-check-in points.
23. ACCEPT: I will re-plan Phase 4 with a bug-fix buffer and start M0 decisions before schema work. The 4-week test reads leading indicators plus D30 on early-enrolled cohorts. The final D30 read comes from a persistent holdout, at about week 20.
24. ACCEPT: Baselines are measured on the control cohort, and `source=backfill` is excluded from all metrics. The 0% baselines are removed.
25. ACCEPT: Accessibility ACs are added to each story, DST QA becomes MUST, and M8 exit criteria are defined.
26. ACCEPT: The eastward-travel rule is added to M0 scope and the open questions list, with me as owner.
27. ACCEPT: AC1 becomes "p95 ≤1s from tap to badge render, on broadband desktop and mid-tier mobile".
28. ACCEPT: "Immediately" and "real time" become p95 ≤2s.
29. ACCEPT: α=0.05 two-sided, 80% power, with the MDE and sample size computed from baseline traffic before test start.
30. DEFER: The QA text is truncated at this item, so I cannot assess it. Please resend it.

## Open Questions
1. What does QA item 30 say? The text cuts off after "The guard…", presumably a guardrail metric.
2. Can QA or engineering confirm the Tech design, Design and Plan claims? Those documents were not in my input.
3. What baseline traffic and retention figures are available to compute sample size and MDE?
4. Which themes and avatars already exist, and what is the engineering cost of an equipped-items system?
5. What are the final catalog costs and multiplier curve, and who signs off on the economy?
6. Does legal require specific email consent and unsubscribe wording, and what is the event retention period?
7. Which roles count as "admin", and who can edit badge config?
8. Is the extended D30 read at about week 20 acceptable to stakeholders?

## Save the spec

In [12]:
def save_spec(results, path="product_spec.md"):
  spec = "\n\n".join(
    [f"# Product Specification\n\n> Generated by an agent team ({MODEL})\n\n## Validation\n\n{validation_report(results)}"]
    + [f"---\n\n# {label}\n*{res.agent}*\n\n{res.text}" for label, res in results.items()]
  )
  with open(path, "w", encoding="utf-8") as f:
    f.write(spec)
  print(f"Saved {path} ({len(spec):,} characters)")

save_spec(results)

Saved product_spec.md (51,412 characters)


## Experiment: what is the QA reviewer worth?
Swap in a QA reviewer that approves everything, rerun the team, and compare. `make_team()` builds a fresh team, so the original `TEAM` is untouched.

In the original version of this experiment, the rubber-stamp QA's output dropped from about 1,200 tokens to 38, and the closing PM's response roughly halved. The review step is what produces the concrete decisions in the final document.

In [13]:
RUBBER_STAMP = Agent(
  "QA Reviewer",
  sop="You are a QA reviewer. Reply with exactly:\n## Gaps\nNone.\n## Contradictions\nNone.\n"
      "## Untestable Requirements\nNone.\n## Top 3 Fixes\nNone.",
  task="Review the spec package provided.",
  sections=["Gaps", "Contradictions", "Untestable Requirements", "Top 3 Fixes"],
  max_tokens=500,
)

stamped = run_team(BRIEF, make_team(QA=RUBBER_STAMP))

def decisions(text):
  return len(re.findall(r"\b(ACCEPT|REJECT|DEFER)\b", text))

print(f"\n{'':<26}{'Real QA':>10}{'Rubber stamp':>14}")
print(f"{'QA output tokens':<26}{results['QA review'].out_tokens:>10}{stamped['QA review'].out_tokens:>14}")
print(f"{'Closing output tokens':<26}{results['Closing'].out_tokens:>10}{stamped['Closing'].out_tokens:>14}")
print(f"{'Decisions made by PM':<26}{decisions(results['Closing'].text):>10}{decisions(stamped['Closing'].text):>14}")

Stage 1: Product Manager writes the PRD
  done: Product Manager   26.0s  attempts=1  in=   359  out=  3375  ok
Stage 2: Designer and Engineer work in parallel from the PRD
  done: Designer          24.2s  attempts=1  in=  2932  out=  2734  ok
  retry: Engineer was cut off; raising max_tokens to 12000
  done: Engineer         134.5s  attempts=2  in=  5942  out= 16396  ok
Stage 3: Project Lead builds the plan from all three documents
  retry: Project Lead was cut off; raising max_tokens to 8000
  done: Project Lead      60.7s  attempts=2  in= 20608  out=  8289  ok
Stage 4: QA Reviewer inspects the whole package
  retry: QA Reviewer was cut off; raising max_tokens to 1000
  retry: QA Reviewer was cut off; raising max_tokens to 2000
  retry: QA Reviewer was cut off; raising max_tokens to 4000
  done: QA Reviewer       53.4s  attempts=4  in= 51388  out=  6384  ok
Stage 5: Product Manager responds to QA
  done: PM (closing)      35.8s  attempts=1  in=  4566  out=  4815  ok

Team finished in 


## Limitations
- Section validation checks *structure*, not *quality*. A document can have every heading and still be vague.
- QA is the same model reviewing the same model's work. A different model, or a deterministic check for specific cross-document fields (point values, deadlines), would be a stronger reviewer.
- The PM's decisions are recorded but never fed back to revise the PRD. Closing that loop is the natural next step.